In [2]:
# ==============================================================================
# STEP 3.4: SYSTEM EVALUATION, PERFORMANCE METRICS, AND LATENCY BENCHMARKING
# ==============================================================================
import os
import time
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

# 1. Path Configuration
WEIGHTS_PATH = r"D:\SmartVision\Object Detection\yolov8n.pt"
DATA_YAML_PATH = r"D:\SmartVision\DataSets\smartvision_dataset\detection\data.yaml"

if not os.path.exists(WEIGHTS_PATH):
    raise FileNotFoundError(f"Weights not found at: {WEIGHTS_PATH}. Please train the model first.")

print(f"🔄 Initializing evaluation suite using weights: {WEIGHTS_PATH}")
model = YOLO(WEIGHTS_PATH)

# 2. Execute Formal Model Validation Suite
print("\n📊 Evaluating model performance across the validation split...")
print("-" * 75)

# Running val generates standard validation charts (confusion matrix, PR curve)
metrics = model.val(
    data=DATA_YAML_PATH,
    split="val",
    imgsz=640,
    batch=16,
    plots=True  # Generates confusion_matrix.png, F1_curve.png, etc.
)

print("-" * 75)
print("✅ Validation split processing complete.")

# 3. Extract Core System Performance Metrics
map50 = metrics.box.map50           # mAP at IoU=0.50
map95 = metrics.box.map             # mAP at IoU=0.50:0.95
mean_precision = metrics.box.mp     # System wide Mean Precision
mean_recall = metrics.box.mr        # System wide Mean Recall

# Compute true hardware frame throughput (FPS)
t_preprocess = metrics.speed['preprocess']   # Latency in ms
t_inference = metrics.speed['inference']     # Latency in ms
t_postprocess = metrics.speed['postprocess'] # Latency in ms
total_latency = t_preprocess + t_inference + t_postprocess
achieved_fps = 1000.0 / total_latency

print("\n🎯 === GLOBAL ACCURACY & OPERATIONAL THROUGHPUT === ")
print(f" 🟢 mean Average Precision (mAP@0.50)  : {map50:.2%}")
print(f" 🟢 mean Average Precision (mAP@0.5:0.95): {map95:.2%}")
print(f" 🟢 Macro Precision (System Mean)       : {mean_precision:.2%}")
print(f" 🟢 Macro Recall (System Mean)          : {mean_recall:.2%}")
print(f" 🟢 End-to-End Image Processing Latency: {total_latency:.2f} ms")
print(f" 🟢 Target System Throughput            : {achieved_fps:.2f} FPS")
print("======================================================\n")

# 4. Generate Per-Class Granular Performance Matrix (Fixed Indexing)
print("📋 === PER-CLASS ACCURACY METRIC PROFILE MATRIX ===")
per_class_data = []

# Loop over the actual indices evaluated by the validation engine
for i, class_id in enumerate(metrics.box.ap_class_index):
    # Map the class_id back to your model's text labels safely
    class_name = model.names.get(class_id, f"UNKNOWN_{class_id}")
    
    # Pass the sequential array position 'i', not the raw 'class_id'
    p = metrics.box.class_result(i)[0]
    r = metrics.box.class_result(i)[1]
    ap50 = metrics.box.class_result(i)[2]
    ap95 = metrics.box.class_result(i)[3]
    
    per_class_data.append({
        "Class ID": class_id,
        "Class Name": class_name.upper(),
        "Precision": f"{p:.2%}",
        "Recall": f"{r:.2%}",
        "mAP@0.50": f"{ap50:.2%}",
        "mAP@0.50:0.95": f"{ap95:.2%}"
    })

df_metrics = pd.DataFrame(per_class_data)
display(df_metrics)

# 5. Pipeline Visualization: Run Predictions & Display Output
print("\n🖼️ === SAMPLE PREDICTION VISUALIZATION PASS ===")
test_image_path = r"D:\SmartVision\DataSets\smartvision_dataset\detection\images\image_001982.jpg"

if os.path.exists(test_image_path):
    # Run test frame tracking
    results = model.predict(source=test_image_path, conf=0.25, imgsz=640, save=False)
    
    # Extract the internal BGR canvas array and map colors to RGB space
    annotated_img = results[0].plot()
    annotated_img_rgb = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)
    
    # Render preview inside the Jupyter canvas
    plt.figure(figsize=(10, 7))
    plt.imshow(annotated_img_rgb)
    plt.axis('off')
    plt.title(f"Validation Preview Matrix: {os.path.basename(test_image_path)}")
    plt.show()
else:
    print(f"⚠️ Target visualization image path target not found at: {test_image_path}")

print(f"\n📁 Assessment logs and analytical artifacts saved to: {os.path.abspath(metrics.save_dir)}")

🔄 Initializing evaluation suite using weights: D:\SmartVision\Object Detection\yolov8n.pt

📊 Evaluating model performance across the validation split...
---------------------------------------------------------------------------
Ultralytics 8.4.41  Python-3.12.5 torch-2.11.0+cpu CPU (Intel Core i3-6006U 2.00GHz)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access  (ping: 0.20.1 ms, read: 2.61.9 MB/s, size: 126.0 KB)
val: Scanning D:\SmartVision\DataSets\smartvision_dataset\detection\labels.cache... 2210 images, 0 backgrounds, 1687 corrupt: 100% ━━━━━━━━━━━━ 2210/2210  0.0s
train: D:\SmartVision\DataSets\smartvision_dataset\detection\images\image_000001.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.0362   1.3258   1.133727 1.02763  1.02652  1.384619]
train: D:\SmartVision\DataSets\smartvision_dataset\detection\images\image_000002.jpg: ignoring corrupt image/label: non-normalized or out of bounds coo

,Class ID,Class Name,Precision,Recall,mAP@0.50,mAP@0.50:0.95
0,0,PERSON,17.36%,9.87%,4.09%,1.47%
1,1,BICYCLE,23.39%,2.94%,1.04%,0.52%
2,2,CAR,7.48%,2.88%,1.34%,0.83%
3,3,MOTORCYCLE,67.49%,42.86%,36.64%,11.17%
4,4,AIRPLANE,15.66%,10.23%,2.99%,0.87%
5,5,BUS,67.71%,39.96%,31.65%,9.41%
6,6,TRAIN,32.71%,26.19%,13.06%,4.20%
7,7,TRUCK,38.38%,8.33%,5.75%,2.53%
8,8,BOAT,0.00%,0.00%,0.00%,0.00%
9,9,TRAFFIC LIGHT,0.00%,0.00%,0.01%,0.00%



🖼️ === SAMPLE PREDICTION VISUALIZATION PASS ===

image 1/1 D:\SmartVision\DataSets\smartvision_dataset\detection\images\image_001982.jpg: 448x640 1 cat, 3 dogs, 2 couchs, 396.1ms
Speed: 4.2ms preprocess, 396.1ms inference, 34.2ms postprocess per image at shape (1, 3, 448, 640)


<Figure size 1000x700 with 1 Axes>


📁 Assessment logs and analytical artifacts saved to: D:\SmartVision\Object Detection\runs\detect\val-3


In [3]:
import os
import cv2

# Set your paths here
IMAGE_DIR = r"D:\SmartVision\DataSets\smartvision_dataset\detection\images"
LABEL_DIR = r"D:\SmartVision\DataSets\smartvision_dataset\detection\labels"

for label_file in os.listdir(LABEL_DIR):
    if not label_file.endswith('.txt') or label_file == 'classes.txt':
        continue
        
    label_path = os.path.join(LABEL_DIR, label_file)
    
    # Find matching image to get its width and height
    base_name = os.path.splitext(label_file)[0]
    img_exts = ['.jpg', '.jpeg', '.png']
    img_path = None
    
    for ext in img_exts:
        test_path = os.path.join(IMAGE_DIR, base_name + ext)
        if os.path.exists(test_path):
            img_path = test_path
            break
            
    if not img_path:
        print(f"Skipping: No image found for {label_file}")
        continue

    # Load image dimensions
    img = cv2.imread(img_path)
    if img is None:
        continue
    h, w, _ = img.shape

    # Read and convert coordinates
    corrected_lines = []
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            
            class_id = parts[0]
            # Assuming format is: class x_center y_center width height
            coords = [float(x) for x in parts[1:]]
            
            # Check if coordinates are already normalized (all less than or equal to 1)
            # If they are greater than 1, they are likely pixel values and need fixing
            if any(c > 1.0 for c in coords):
                # Normalizing pixel coordinates
                x_center = coords[0] / w
                y_center = coords[1] / h
                box_width = coords[2] / w
                box_height = coords[3] / h
                
                # Clip values strictly between 0 and 1 just in case of slight rounding overflows
                x_center = min(max(x_center, 0.0), 1.0)
                y_center = min(max(y_center, 0.0), 1.0)
                box_width = min(max(box_width, 0.0), 1.0)
                box_height = min(max(box_height, 0.0), 1.0)
                
                corrected_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {box_width:.6f} {box_height:.6f}\n")
            else:
                # Keep line as-is if already normalized
                corrected_lines.append(line)

    # Overwrite label file with correct format
    with open(label_path, 'w') as f:
        f.writelines(corrected_lines)

print("Normalization complete! Try running your YOLOv8 validation suite again.")

Normalization complete! Try running your YOLOv8 validation suite again.


In [4]:
from ultralytics import YOLO

# Load your model (use your trained weights path if you aren't using the base model)
model = YOLO('D:\SmartVision\Object Detection\yolov8n.pt') 

# Run validation
metrics = model.val(data=r"D:\SmartVision\DataSets\smartvision_dataset\detection\data.yaml")

<>:4: SyntaxWarning: invalid escape sequence '\S'
<>:4: SyntaxWarning: invalid escape sequence '\S'
C:\Users\Windows 10\AppData\Local\Temp\ipykernel_13540\1487810189.py:4: SyntaxWarning: invalid escape sequence '\S'
  model = YOLO('D:\SmartVision\Object Detection\yolov8n.pt')


Ultralytics 8.4.41  Python-3.12.5 torch-2.11.0+cpu CPU (Intel Core i3-6006U 2.00GHz)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access  (ping: 0.50.6 ms, read: 1.00.4 MB/s, size: 160.7 KB)
val: Scanning D:\SmartVision\DataSets\smartvision_dataset\detection\labels... 2210 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2210/2210 48.5it/s 45.6s<0.1s
val: New cache created: D:\SmartVision\DataSets\smartvision_dataset\detection\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 139/139 4.3s/it 10:024.7s
                   all       2210      22163     0.0906     0.0215     0.0093    0.00281
                person       1188       6502      0.153     0.0646     0.0274    0.00824
               bicycle        167        656      0.138      0.029    0.00613    0.00252
                   car        454       2187     0.0518     0.0119     0.0019   0.000302
          